In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,RandomizedSearchCV,TimeSeriesSplit
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(r"data/Merged_Train_Data.csv")

In [3]:
df.head()

,Unnamed: 0,year,month,day,hour,O3_forecast,NO2_forecast,T_forecast,q_forecast,u_forecast,v_forecast,w_forecast,NO2_satellite,HCHO_satellite,ratio_satellite,O3_target,NO2_target,location
0,0,2022.0,7.0,28.0,0.0,73.35,57.54,22.99,7.68,-3.46,1.28,1.02,NaN,NaN,NaN,5.03,6.75,Satyawati College
1,1,2022.0,7.0,28.0,1.0,82.77,57.25,23.90,7.70,-1.35,0.29,0.99,NaN,NaN,NaN,5.08,6.07,Satyawati College
2,2,2022.0,7.0,28.0,2.0,92.19,56.97,23.89,7.72,0.76,-0.70,0.96,NaN,NaN,NaN,6.95,3.38,Satyawati College
3,3,2022.0,7.0,28.0,3.0,101.62,56.68,23.92,7.74,2.87,-1.69,0.93,NaN,NaN,NaN,5.80,4.85,Satyawati College
4,4,2022.0,7.0,28.0,4.0,113.51,64.06,25.55,7.81,2.45,-1.15,0.66,NaN,NaN,NaN,7.22,7.72,Satyawati College


In [4]:
df.columns

Index(['Unnamed: 0', 'year', 'month', 'day', 'hour', 'O3_forecast',
       'NO2_forecast', 'T_forecast', 'q_forecast', 'u_forecast', 'v_forecast',
       'w_forecast', 'NO2_satellite', 'HCHO_satellite', 'ratio_satellite',
       'O3_target', 'NO2_target', 'location'],
      dtype='object')

In [5]:
df.rename(columns={
    "T_forecast": "Temperature_forecast",
    "q_forecast": "Specific_humidity_forecast",
    "u_forecast": "U_wind_forecast",
    "v_forecast": "V_wind_forecast",
    "w_forecast": "Vertical_wind_forecast"
},inplace=True)

print(df.columns)

Index(['Unnamed: 0', 'year', 'month', 'day', 'hour', 'O3_forecast',
       'NO2_forecast', 'Temperature_forecast', 'Specific_humidity_forecast',
       'U_wind_forecast', 'V_wind_forecast', 'Vertical_wind_forecast',
       'NO2_satellite', 'HCHO_satellite', 'ratio_satellite', 'O3_target',
       'NO2_target', 'location'],
      dtype='object')


In [6]:
df = df.drop(columns=["Unnamed: 0","NO2_satellite","HCHO_satellite","ratio_satellite"])
df.head()

,year,month,day,hour,O3_forecast,NO2_forecast,Temperature_forecast,Specific_humidity_forecast,U_wind_forecast,V_wind_forecast,Vertical_wind_forecast,O3_target,NO2_target,location
0,2022.0,7.0,28.0,0.0,73.35,57.54,22.99,7.68,-3.46,1.28,1.02,5.03,6.75,Satyawati College
1,2022.0,7.0,28.0,1.0,82.77,57.25,23.90,7.70,-1.35,0.29,0.99,5.08,6.07,Satyawati College
2,2022.0,7.0,28.0,2.0,92.19,56.97,23.89,7.72,0.76,-0.70,0.96,6.95,3.38,Satyawati College
3,2022.0,7.0,28.0,3.0,101.62,56.68,23.92,7.74,2.87,-1.69,0.93,5.80,4.85,Satyawati College
4,2022.0,7.0,28.0,4.0,113.51,64.06,25.55,7.81,2.45,-1.15,0.66,7.22,7.72,Satyawati College


In [7]:
df["datetime"] = pd.to_datetime(df[["year","month","day","hour"]])
df = df.sort_values(by=["location","datetime"])

In [8]:
lag_cols = [
    "Temperature_forecast",
    "O3_forecast",
    "NO2_forecast",
    "Specific_humidity_forecast"
]

for col in lag_cols:
    df[f"{col}_lag_1"] = df.groupby("location")[col].shift(1)
    df[f"{col}_lag_3"] = df.groupby("location")[col].shift(3)
    df[f"{col}_lag_6"] = df.groupby("location")[col].shift(6)
    df[f"{col}_lag_24"] = df.groupby("location")[col].shift(24)


for col in lag_cols:
    df[f"{col}_roll_mean_6"] = df.groupby("location")[col].rolling(6).mean().reset_index(level=0, drop=True)
    df[f"{col}_roll_mean_24"] = df.groupby("location")[col].rolling(24).mean().reset_index(level=0, drop=True)
    
    df[f"{col}_roll_std_24"] = df.groupby("location")[col].rolling(24).std().reset_index(level=0, drop=True)


df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


In [9]:
df.head()

,year,month,day,hour,O3_forecast,NO2_forecast,Temperature_forecast,Specific_humidity_forecast,U_wind_forecast,V_wind_forecast,...,NO2_forecast_roll_mean_6,NO2_forecast_roll_mean_24,NO2_forecast_roll_std_24,Specific_humidity_forecast_roll_mean_6,Specific_humidity_forecast_roll_mean_24,Specific_humidity_forecast_roll_std_24,hour_sin,hour_cos,month_sin,month_cos
92884,2019.0,7.0,11.0,0.0,0.79,85.47,12.41,19.28,-3.11,1.14,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000,-0.5,-0.866025
92885,2019.0,7.0,11.0,1.0,0.93,93.49,13.36,19.46,-2.50,0.70,...,NaN,NaN,NaN,NaN,NaN,NaN,0.258819,0.965926,-0.5,-0.866025
92886,2019.0,7.0,11.0,2.0,1.07,101.51,15.49,19.64,-1.90,0.26,...,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,0.866025,-0.5,-0.866025
92887,2019.0,7.0,11.0,3.0,1.21,109.53,16.53,19.82,-1.29,-0.18,...,NaN,NaN,NaN,NaN,NaN,NaN,0.707107,0.707107,-0.5,-0.866025
92888,2019.0,7.0,11.0,4.0,0.81,112.99,13.77,20.60,-1.63,-0.50,...,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,0.500000,-0.5,-0.866025


In [10]:
df = df.dropna()

In [11]:
X = df.drop(columns=["O3_target","NO2_target","datetime"])
y = df[["O3_target","NO2_target"]]

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [13]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

num_transformer = StandardScaler()
cat_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", cat_transformer, cat_features),
        ("StandardScaler", num_transformer, num_features),        
    ]
)

In [14]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [15]:
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = root_mean_squared_error(true,predicted)
    r2 = r2_score(true,predicted)

    return mae,mse,rmse,r2

In [16]:
models = {
    "Linear Regression": LinearRegression(n_jobs=-1),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "KNN": KNeighborsRegressor(n_jobs=-1),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_jobs=-1),
    "Adaboost": AdaBoostRegressor(),
    "XgBoost": XGBRegressor(n_jobs=-1)
}

model_list = []
mae_list = []
mse_list = []
rmse_list = []
r2_list = []

for name, model in models.items():
    if name in ["Adaboost","XgBoost"]:
        model = MultiOutputRegressor(model)

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae, train_mse, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_mse, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    print(name)
    model_list.append(name)

    print("Model performance for Training set")
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- Mean Squared Error: {:.4f}".format(train_mse))
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- R2 Score: {:.4f}".format(train_r2))

    print("-" * 35)

    print("Model performance for Test set")
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- Mean Squared Error: {:.4f}".format(test_mse))
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- R2 Score: {:.4f}".format(test_r2))

    mae_list.append(test_mae)
    mse_list.append(test_mse)
    rmse_list.append(test_rmse)
    r2_list.append(test_r2)

    print("=" * 35)
    print("\n")

Linear Regression
Model performance for Training set
- Mean Absolute Error: 17.4091
- Mean Squared Error: 582.0638
- Root Mean Squared Error: 24.0934
- R2 Score: 0.3828
-----------------------------------
Model performance for Test set
- Mean Absolute Error: 17.4337
- Mean Squared Error: 585.0560
- Root Mean Squared Error: 24.1611
- R2 Score: 0.3814


Ridge
Model performance for Training set
- Mean Absolute Error: 17.4090
- Mean Squared Error: 582.0638
- Root Mean Squared Error: 24.0934
- R2 Score: 0.3828
-----------------------------------
Model performance for Test set
- Mean Absolute Error: 17.4337
- Mean Squared Error: 585.0548
- Root Mean Squared Error: 24.1610
- R2 Score: 0.3814


Lasso
Model performance for Training set
- Mean Absolute Error: 17.8641
- Mean Squared Error: 627.9541
- Root Mean Squared Error: 25.0297
- R2 Score: 0.3334
-----------------------------------
Model performance for Test set
- Mean Absolute Error: 17.8697
- Mean Squared Error: 629.9404
- Root Mean Square

In [17]:
results = pd.DataFrame(list(zip(model_list,mae_list,mse_list,rmse_list,r2_list)),columns=["Model","MAE","MSE","RMSE","R2"])
results

,Model,MAE,MSE,RMSE,R2
0,Linear Regression,17.433731,585.055994,24.161054,0.381434
1,Ridge,17.433679,585.054838,24.161031,0.381435
2,Lasso,17.869737,629.940403,25.074558,0.333327
3,KNN,8.432716,182.375586,13.478112,0.808051
4,Decision Tree,8.296381,245.530653,15.669418,0.736834
5,Random Forest,6.686346,120.646013,10.983614,0.870418
6,Adaboost,24.894905,866.375750,29.351631,0.091397
7,XgBoost,10.123604,218.909821,14.774984,0.768933


In [18]:
rf_params = {
    "n_estimators": [200,300,400],
    "max_depth": [None,10,20,30],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4],
    "max_features": ["sqrt","log2",None]
}

xgb_params = {
    "estimator__n_estimators": [200,400,600],
    "estimator__max_depth": [4,6,8],
    "estimator__learning_rate": [0.03,0.05,0.1],
    "estimator__subsample": [0.8,1.0],
    "estimator__colsample_bytree": [0.8,1.0]
}

rf = RandomForestRegressor(n_jobs=1,random_state=42)
xgb = XGBRegressor(tree_method="hist",n_jobs=1,random_state=42)
xgb_multi = MultiOutputRegressor(xgb)

In [19]:
randomcv_models = [
    ("Random Forest",rf,rf_params),
    ("XGBoost",xgb_multi,xgb_params)               
]

In [20]:
model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,
        cv=TimeSeriesSplit(3),
        verbose=2,
        n_jobs=-1,
        scoring="r2",
        random_state=42
    )
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
---------------- Best Params for Random Forest -------------------
{'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 30}
---------------- Best Params for XGBoost -------------------
{'estimator__subsample': 0.8, 'estimator__n_estimators': 600, 'estimator__max_depth': 8, 'estimator__learning_rate': 0.1, 'estimator__colsample_bytree': 1.0}


In [23]:
save_path = "../artifacts"
os.makedirs(save_path,exist_ok=True)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=400,
        min_samples_split=5,
        min_samples_leaf=1,
        max_features=None,
        max_depth=30,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": MultiOutputRegressor(
        XGBRegressor(
            n_estimators=600,
            max_depth=8,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=42
        )
    )
}

for name,model in models.items():

    model.fit(X_train,y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    model_train_mae,model_train_mse,model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    model_test_mae,model_test_mse,model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)

    print(name)
    print('Model performance for Training set')
    print("- Mean Squared Error: {:.4f}".format(model_train_mse))
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))
    print()
    print('Model performance for Test set')
    print("- Mean Squared Error: {:.4f}".format(model_test_mse))
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    print("="*50)
    print()

    file_path = os.path.join(save_path,f"{name.replace(' ','_')}_V2.pkl")

    with open(file_path,"wb") as f:
        pickle.dump(model,f)

    print(f"{name} saved at {file_path}")
    print("--------------------------------------------------\n")


with open(os.path.join(save_path,"preprocessor_V2.pkl"),"wb") as f:
    pickle.dump(preprocessor,f)

print("Preprocessor saved successfully.")

Random Forest
Model performance for Training set
- Mean Squared Error: 27.8740
- Root Mean Squared Error: 5.2796
- Mean Absolute Error: 3.2366
- R2 Score: 0.9700

Model performance for Test set
- Mean Squared Error: 126.0201
- Root Mean Squared Error: 11.2257
- Mean Absolute Error: 6.9297
- R2 Score: 0.8647

Random Forest saved at ../artifacts\Random_Forest_V2.pkl
--------------------------------------------------

XGBoost
Model performance for Training set
- Mean Squared Error: 42.7905
- Root Mean Squared Error: 6.5326
- Mean Absolute Error: 4.5857
- R2 Score: 0.9546

Model performance for Test set
- Mean Squared Error: 118.0965
- Root Mean Squared Error: 10.8500
- Mean Absolute Error: 7.2442
- R2 Score: 0.8755

XGBoost saved at ../artifacts\XGBoost_V2.pkl
--------------------------------------------------

Preprocessor saved successfully.
